# Regression using Artificial Neural Networks

using the doped feature dataset

In [ ]:
# Import standard Libraries
import pandas as pd
import seaborn as sns
import altair as alt
import tensorflow as tf
import matplotlib.pyplot as plt


pd.set_option('display.float_format', '{:.4f}'.format)

sns.set(rc={'figure.figsize':(10,10)})
print("imports ok")

##Load Data

In [ ]:
# Load data file
data = pd.read_csv('C:\\Users\\DD\\Desktop\\IAAC\\DATA encoding\\final project_entropy prediction\\data\\processed\\dataset_dropped_features_clean.csv')
pd.options.display.max_columns = None

print(data)

In [ ]:
print(data.info())

In [ ]:
for colname, col in data.items():
  print(colname, "min_val", col.min(), "max_val", col.max())

In [ ]:
data.describe()

In [ ]:
print(data.columns.tolist())


we can see that some predictors are binary, while others are not

In [ ]:
feature_cols = ['n_3way', 'n_deadend', 'proportion_4way', 'proportion_3way',
                'proportion_deadend', 'mean_edge_length', 'total_edge_length', 'circuity']

print("Features:", feature_cols)
print("Target: entropy_normalised")
data[feature_cols].describe()


In [ ]:
data_numerical = data[['entropy_normalised', 'mean_edge_length', 'total_edge_length', 'circuity']]
sns.pairplot(data_numerical)


We can see there is a direct relationship between age and experience, but not so much between wage and education, there is a non-linearity

##Prepare Data

**NORMALIZE INPUTS**

In [ ]:
#same code as last time

feature_cols = ['n_3way', 'n_deadend', 'proportion_4way', 'proportion_3way',
                'proportion_deadend', 'mean_edge_length', 'total_edge_length', 'circuity']

X = data[feature_cols].to_numpy()

from sklearn.preprocessing import StandardScaler
scalerX = StandardScaler()

X_scaled = scalerX.fit_transform(X)

print(X_scaled.shape)


In [ ]:
#declare regression target
y = data.loc[:, "entropy_normalised"].to_numpy()

y = y.reshape(-1, 1)

from sklearn.preprocessing import MinMaxScaler
scalerY = MinMaxScaler()

# Apply the scaler to our Y-features
y_scaled = scalerY.fit_transform(y)

print(y_scaled.shape)



**SPLIT INTO TRAIN AND TEST**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size = 0.2, random_state = 21)

In [ ]:
#visualize our data
#we can see that scikitlearn doesnt care if it is a dataframe or a numpy array, because they all function on the same way
print("TRAIN", "input", X_train.shape, "output", y_train.shape)
print("TEST", "input", X_test.shape, "output", y_test.shape)


#Build model

From the cheatSheet
Regression between 0 and 1>>
      activation = relu for hidden layers / sigmoid for final layer
      loss = mean squared error
      optimizer = adam
      input from data, is 8 columns
      output is 1 value prediction

In [ ]:
# Instantiate a sequential model
model = tf.keras.models.Sequential()
n_cols = X_scaled.shape[1]

model.add(tf.keras.layers.Input(shape=(n_cols,)))
model.add(tf.keras.layers.Dense(16, activation='relu'))
model.add(tf.keras.layers.Dense(8, activation='relu'))
model.add(tf.keras.layers.Dense(4, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='mean_squared_error')



In [ ]:
model.summary()

#Train model

In [ ]:
# Fit your model to the training data for 200 epochs
#we assign this to history variable so we can plot the training data
history = model.fit(X_train,y_train,epochs=500, validation_split=0.2)

In [ ]:
# summarize history for accuracy
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('loss function')
plt.ylabel('mse')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

#Evaluate model on test data



In [ ]:
# Evaluate your model accuracy on the test data
loss_test = model.evaluate(X_test,y_test)

# Print accuracy
print('mse_test:', loss_test)

#Plot error


In [ ]:
y_pred = scalerY.inverse_transform(model.predict(X_test))
y_truth = scalerY.inverse_transform(y_test)

plt.scatter(y_truth, y_pred)
plt.ylim((0, 1))
plt.xlim((0, 1))
plt.axline((0, 0), slope=1, ls="--")

plt.xlabel("y_truth (entropy normalised)")
plt.ylabel("y_pred (entropy normalised)")



In [ ]:
def plot_comparison(x_val, pred, truth, xlab, ylab):
  fig, ax1 = plt.subplots(figsize = (10,10))
  ax1.plot(x_val, truth, color = "red", label = "truth",linestyle='None', marker = "o", markersize = 5)
  ax1.plot(x_val, pred, color = "blue", label = "pred",linestyle='None', marker = "o", markersize = 4, alpha = 0.5)

  ax1.set_xlabel(xlab)
  ax1.set_ylabel(ylab)
  ax1.legend()
  plt.title('Prediction Comparison')
  plt.show()

In [ ]:
mean_edge_test = scalerX.inverse_transform(X_test)[:, feature_cols.index('mean_edge_length')]
circuity_test = scalerX.inverse_transform(X_test)[:, feature_cols.index('circuity')]

plot_comparison(mean_edge_test, y_pred, y_truth, "mean edge length", "entropy normalised")
plot_comparison(circuity_test, y_pred, y_truth, "circuity", "entropy normalised")


In [ ]:
error = y_pred - y_truth
plt.hist(error, bins=25)
plt.xlabel('Prediction Error [entropy normalised]')
_ = plt.ylabel('Count')


In [ ]:
model.save('ANNregression_01.keras')

import pickle
pickle.dump(scalerX, open('scalerX.pkl', 'wb'))
pickle.dump(scalerY, open('scalerY.pkl', 'wb'))

print("Model and scalers saved!")
